In [4]:
import os
import requests
from time import sleep

from zoneinfo import ZoneInfo
from astral import LocationInfo
from astral.sun import sun
from datetime import datetime, time, date, timedelta

import os
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from time import sleep
from dateutil import parser
from dotenv import load_dotenv
import json

import numpy as np
from collections import defaultdict, Counter

import drive

In [28]:
lat=latitude = '37.4277'
lon=longitude = '-122.1701'
user_agent=USER_AGENT = {"User-Agent": "Pikachu"}
tzone= ZoneInfo("America/Los_Angeles")

In [37]:
end=datetime.now(tzone)
start= datetime.now(tzone) - timedelta(hours=6)

observe=drive.get_observation(lat, lon, start, end, limit=100)
single=observe['features'][1]['properties']

In [49]:
single=observe['features'][6]['properties']
time= single['timestamp']
weather = single['textDescription']
temperature= single['temperature']['value']
windDirection = single['windDirection']['value']
windSpeed  =  single['windSpeed']['value']
humidity = single['relativeHumidity']['value']

In [51]:
single

{'@id': 'https://api.weather.gov/stations/AW020/observations/2026-01-06T08:25:00+00:00',
 '@type': 'wx:ObservationStation',
 'elevation': {'unitCode': 'wmoUnit:m', 'value': 10},
 'station': 'https://api.weather.gov/stations/AW020',
 'stationId': 'AW020',
 'stationName': 'AE6EO MOUNTAIN VIEW',
 'timestamp': '2026-01-06T08:25:00+00:00',
 'rawMessage': '',
 'textDescription': '',
 'icon': None,
 'presentWeather': [],
 'temperature': {'unitCode': 'wmoUnit:degC',
  'value': 11.67,
  'qualityControl': 'V'},
 'dewpoint': {'unitCode': 'wmoUnit:degC',
  'value': 10.39,
  'qualityControl': 'V'},
 'windDirection': {'unitCode': 'wmoUnit:degree_(angle)',
  'value': 126,
  'qualityControl': 'V'},
 'windSpeed': {'unitCode': 'wmoUnit:km_h-1',
  'value': 3.204,
  'qualityControl': 'V'},
 'windGust': {'unitCode': 'wmoUnit:km_h-1',
  'value': 8.064,
  'qualityControl': 'S'},
 'barometricPressure': {'unitCode': 'wmoUnit:Pa',
  'value': 101610,
  'qualityControl': 'V'},
 'seaLevelPressure': {'unitCode': 'w

In [52]:
import requests
from time import sleep

USER_AGENT = {"User-Agent": "Pikachu"}

def get_station_ids_near(lat, lon, max_stations=15):
    # 1) points -> observationStations
    points = requests.get(
        f"https://api.weather.gov/points/{lat},{lon}",
        headers=USER_AGENT,
        timeout=20
    ).json()

    sleep(1)

    stations_url = points["properties"]["observationStations"]

    # 2) observationStations -> stations list
    stations = requests.get(
        stations_url,
        headers=USER_AGENT,
        timeout=20
    ).json()

    feats = stations.get("features", [])
    station_ids = []
    for f in feats[:max_stations]:
        sid = f.get("properties", {}).get("stationIdentifier")
        if sid:
            station_ids.append(sid)
    return station_ids


def latest_obs_props(station_id):
    obs = requests.get(
        f"https://api.weather.gov/stations/{station_id}/observations/latest",
        headers=USER_AGENT,
        timeout=20
    ).json()
    return obs.get("properties", {}) or {}


def score_station(props):
    # 字段“有值”的数量 + rawMessage长度（机场METAR通常更长）
    non_null = sum(1 for _, v in props.items() if v not in (None, "", [], {}))
    raw_len = len(props.get("rawMessage") or "")
    return non_null * 10 + raw_len, non_null, raw_len


def pick_best_station(lat, lon, max_stations=15):
    station_ids = get_station_ids_near(lat, lon, max_stations=max_stations)

    best_id = None
    best = None  # (score, non_null, raw_len)

    for sid in station_ids:
        sleep(1)
        props = latest_obs_props(sid)
        s = score_station(props)
        if best is None or s[0] > best[0]:
            best = s
            best_id = sid

    return best_id, best, station_ids


# ===== 用法 =====
lat, lon = 37.39, -122.08
best_id, best_stats, all_ids = pick_best_station(lat, lon, max_stations=15)
# print("best station:", best_id, "stats(score,non_null,raw_len):", best_stats)
# print("candidates:", all_ids)


In [54]:
params = {"start": start, "end": end, "limit": 100}
obs = requests.get(
        f"https://api.weather.gov/stations/{best_id}/observations",
        headers=user_agent,
        params=params
    ).json()

In [55]:
single=obs['features'][6]['properties']
time= single['timestamp']
weather = single['textDescription']
temperature= single['temperature']['value']
windDirection = single['windDirection']['value']
windSpeed  =  single['windSpeed']['value']
humidity = single['relativeHumidity']['value']

In [56]:
single

{'@id': 'https://api.weather.gov/stations/KHAF/observations/2026-01-06T07:15:00+00:00',
 '@type': 'wx:ObservationStation',
 'elevation': {'unitCode': 'wmoUnit:m', 'value': 20},
 'station': 'https://api.weather.gov/stations/KHAF',
 'stationId': 'KHAF',
 'stationName': 'Half Moon Bay Airport',
 'timestamp': '2026-01-06T07:15:00+00:00',
 'rawMessage': 'KHAF 060715Z AUTO 13009KT 1 1/2SM UP BR BKN018 OVC024 12/12 A2999 RMK AO2',
 'textDescription': 'Cloudy',
 'icon': 'https://api.weather.gov/icons/land/night/ovc?size=medium',
 'presentWeather': [],
 'temperature': {'unitCode': 'wmoUnit:degC',
  'value': 12,
  'qualityControl': 'V'},
 'dewpoint': {'unitCode': 'wmoUnit:degC', 'value': 12, 'qualityControl': 'V'},
 'windDirection': {'unitCode': 'wmoUnit:degree_(angle)',
  'value': 130,
  'qualityControl': 'V'},
 'windSpeed': {'unitCode': 'wmoUnit:km_h-1',
  'value': 16.56,
  'qualityControl': 'V'},
 'windGust': {'unitCode': 'wmoUnit:km_h-1',
  'value': None,
  'qualityControl': 'Z'},
 'barometr